### Trabalho Fase 1 do Curso de Pos-Graduacao FIAP IA para Devs
#### Parte 1 - Dados Gerais - Tratamento de dados SISCAN/Mamografia

Fonte de dados escolhida: DATASUS/SISCAN  
Tipo de dados de origem nesta etapa: Parquet

Arquivo utilizado: `SISCAN_MAMOGRAFIA_RJ_2025.parquet`

Este notebook executa o tratamento inicial da base SISCAN/Mamografia do Rio de Janeiro em 2025. A base ja foi baixada, filtrada para residentes do RJ e convertida para Parquet nas etapas anteriores.

---

## Indice / Sumario da Parte 1

**Objetivo geral - Tratamento e preparacao da base de mamografia**

Organizar a base SISCAN/Mamografia de 2025, avaliar sua estrutura, tratar campos vazios, criar variaveis auxiliares para analise de triagem/rastreamento de cancer de mama e gerar uma versao tratada em Parquet para as proximas etapas.

**Item 1 - Leitura estrutural da base Parquet**

- **1.1 - Carregamento e amostra inicial**: leitura do Parquet e visualizacao inicial dos registros.
- **1.2 - Dimensoes da base**: conferencia de linhas e colunas.
- **1.3 - Tipos de dados e uso de memoria**: avaliacao dos tipos carregados e consumo de memoria.
- **1.4 - Descricao estatistica geral**: resumo estatistico das variaveis disponiveis.

**Item 2 - Dicionario operacional da base SISCAN/Mamografia**

- Organizacao das colunas principais por papel analitico, separando identificacao, perfil, datas, indicacao, achados, BI-RADS e prazos.

**Item 3 - Verificacao de nulos, vazios e consistencia basica**

- **3.1 - Avaliacao de nulos e strings vazias**: identificacao de campos ausentes e strings vazias.
- **3.2 - Colunas totalmente vazias**: mapeamento de colunas sem informacao util.
- **3.3 - Consistencia do identificador e da UF de residencia**: validacao de duplicidade e escopo geografico.

**Item 4 - Tratamento e criacao de variaveis auxiliares**

- **4.1 - Padronizacao de vazios e remocao de colunas sem informacao**: substituicao de vazios por nulos e descarte de colunas completamente vazias.
- **4.2 - Tratamento especifico de `CO_TEMPO_MAMO_ANTERIOR`**: imputacao por regra logica e preenchimento explicito de nao informados.
- **4.3 - Conversoes numericas e temporais**: criacao de versoes numericas e temporais para campos relevantes.
- **4.4 - Variaveis auxiliares para triagem/rastreamento e BI-RADS**: criacao de indicadores derivados para analise posterior.
- **4.5 - Distribuicoes principais apos tratamento**: conferencia das distribuicoes mais relevantes apos as transformacoes.

**Item 5 - Validacao e gravacao da base tratada**

- **5.1 - Shape final e colunas criadas**: conferencia da base final e das variaveis auxiliares.
- **5.2 - Validacoes finais**: checagens de consistencia antes da gravacao.
- **5.3 - Gravacao do Parquet tratado**: geracao de `SISCAN_MAMOGRAFIA_RJ_2025_tratado.parquet`.

**Saida principal da Parte 1**

- Base tratada: `SISCAN_MAMOGRAFIA_RJ_2025_tratado.parquet`
- Shape final validado: 267473 linhas e 93 colunas


#### Item 1 - Leitura estrutural da base Parquet

**Neste passo vamos:**

- carregar o arquivo `SISCAN_MAMOGRAFIA_RJ_2025.parquet`;
- visualizar uma amostra inicial;
- conferir dimensoes, tipos e uso de memoria;
- observar uma descricao estatistica geral da base.


##### 1.1 - Carregamento e amostra inicial

Leitura do arquivo Parquet localizado na raiz deste diretorio.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)


In [2]:
arquivo_origem = Path('SISCAN_MAMOGRAFIA_RJ_2025.parquet')
assert arquivo_origem.exists(), f'Arquivo nao encontrado: {arquivo_origem.resolve()}'

dados = pd.read_parquet(arquivo_origem)
dados.head()

,CO_SEQ_SISCAN_MAMOGRAFIA_RESID,CO_UF_UNIDADE_SAUDE,CO_UF_RESIDENCIA,CO_UF_PREST_SERVICO,CO_MUN_UNIDADE_SAUDE,CO_MUN_RESIDENCIA,CO_MUN_PREST_SERVICO,CO_UNIDADE_SAUDE,CO_PREST_SERVICO,NU_ANO_COMPETENCIA,NU_ANO_MES_COMPETENCIA,CO_RACA_COR,CO_ETNIA,CO_IDADE_PACIENTE,CO_ESCOLARIDADE,TP_RESP_APRES_RISC_ELEV_CANCER,TP_RESP_ANT_MAMA_EXA_PROF_SAUD,TP_RESP_FEZ_MAMOGRA_ALGUMA_VEZ,CO_IND_CLINICA,TP_MAMOGRAFIA_RASTREAMENT,TP_MAMA_PELE_ESQ,TP_MAMA_PELE_DIR,TP_MAMA_ESQ,TP_MAMA_DIR,TP_LINFONODO_ESQ,TP_LINFONODO_DIR,TP_RECOMENDACAO_ESQUERDA,TP_RECOMENDACAO_DIREITA,CO_RECOMENDACAO,NU_CONTROLE_CATEGORIA3,NU_LES_DIAG_CANCER,NU_DIAG_AVAL_QT,NU_REV_MAMOGRAFIA,NU_LES_BIOSPIA_PAAF,NU_NAO_FEZ_CIRURGIAS,NU_FEZ_CIRURGIA_ESQ,NU_FEZ_CIRURGIA_DIR,NU_NODULO_DENS_GORD_ME,NU_NODULO_CALCIFIC_ME,NU_NODULO_DEN_HETEROG_ME,NU_CALC_VASCULARES_ME,NU_CALC_TIPICA_BENIG_ME,NU_LINFON_INTRAMAM_ME,NU_DIS_ARQ_POR_CIR_ME,NU_IMPLANT_SEM_SIN_RUPTU_ME,NU_IMP_COM_SIN_RUPTU_ME,NU_CISTO_OLEOSO_ME,NU_GINECOMASTIA_ME,NU_ECTASIA_DUCTAL_ME,NU_NODULO_DENS_GORD_MD,NU_NODULO_CALCIFIC_MD,NU_NODULO_DEN_HETEROG_MD,NU_CALC_VASCULARES_MD,NU_CALC_TIPICA_BENIG_MD,NU_LINFON_INTRAMAM_MD,NU_DIS_ARQ_POR_CIR_MD,NU_IMPLANT_SEM_SIN_RUPTU_MD,NU_IMP_COM_SIN_RUPTU_MD,NU_CISTO_OLEOSO_MD,NU_GINECOMASTIA_MD,NU_ECTASIA_DUCTAL_MD,SG_SEXO,CO_TEMPO_EXAME,CO_INTERVALO_RESULTADO,CO_INTERVALO_SOLICITACAO,CO_TAMANHO_NOD_ESQ,CO_TAMANHO_NOD_DIR,NU_DIAG_ACHADOS,CO_TEMPO_MAMO_ANTERIOR,CO_BIRADS,NU_TAMANHO_NODULO,ST_ASSIMETRIA_FOCAL,ST_ASSIMETRIA_DIFUSA,ST_DISTORCAO_FOCAL,ST_AREA_DENSA,ST_ACHADO_BENIGNO,ST_TEM_NODULO,TP_NODULO_CAROCO_MAMA,ST_TEM_MICROCALCIFICACAO,CO_ANO_RESULTADO,CO_ANO_MES_RESULTADO,SG_UF_RESIDENCIA
0,296633666,,33,,,330455,,,,2025,202510,01,,055,,02,01,02,02,01,01,01,01,04,01,01,2,2,2,,,0,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,F,2,2,3,,,0,,2,0,N,N,N,N,N,N,04,N,2025,202510,RJ
1,296633687,,33,,,330630,,,,2025,202501,01,,054,,02,01,01,02,01,01,01,03,03,01,03,2,2,2,,,0,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,F,3,4,2,,,0,1,2,0,N,N,N,N,N,N,04,N,2025,202501,RJ
2,296633755,,33,,,330070,,,,2025,202511,01,,045,,02,01,01,02,01,01,01,03,03,03,01,,,0,,,0,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,F,1,3,1,,,0,4,2,0,N,N,N,N,N,N,03,N,2025,202511,RJ
3,296633764,,33,,,330190,,,,2025,202510,03,,052,,03,03,01,02,01,01,01,01,01,01,01,2,2,2,,,0,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,F,1,1,1,,,0,0,2,0,N,N,N,N,N,N,04,N,2025,202510,RJ
4,296637612,,33,,,330240,,,,2025,202502,04,,040,,02,03,03,02,01,01,01,02,02,01,01,,,0,,,0,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,F,3,1,4,,,0,,2,0,N,N,N,N,N,N,04,N,2025,202502,RJ


##### 1.2 - Dimensoes da base

Registro do volume original antes dos tratamentos.

In [3]:
dados.shape

(267473, 82)

##### 1.3 - Tipos de dados e uso de memoria

Como o CSV foi convertido preservando os campos como texto, a maior parte das colunas chega como `string/object`. Conversoes numericas serao feitas apenas nas variaveis usadas na analise.

In [4]:
dados.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 267473 entries, 0 to 267472
Data columns (total 82 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   CO_SEQ_SISCAN_MAMOGRAFIA_RESID  267473 non-null  str  
 1   CO_UF_UNIDADE_SAUDE             267473 non-null  str  
 2   CO_UF_RESIDENCIA                267473 non-null  str  
 3   CO_UF_PREST_SERVICO             267473 non-null  str  
 4   CO_MUN_UNIDADE_SAUDE            267473 non-null  str  
 5   CO_MUN_RESIDENCIA               267473 non-null  str  
 6   CO_MUN_PREST_SERVICO            267473 non-null  str  
 7   CO_UNIDADE_SAUDE                267473 non-null  str  
 8   CO_PREST_SERVICO                267473 non-null  str  
 9   NU_ANO_COMPETENCIA              267473 non-null  str  
 10  NU_ANO_MES_COMPETENCIA          267473 non-null  str  
 11  CO_RACA_COR                     267473 non-null  str  
 12  CO_ETNIA                        267473 non-null  str  


##### 1.4 - Descricao estatistica geral

Avaliacao geral das colunas para entender frequencias, valores predominantes e possiveis inconsistencias iniciais.

In [5]:
dados.describe(include='all').T

,count,unique,top,freq
CO_SEQ_SISCAN_MAMOGRAFIA_RESID,267473,267473,296633666,1
CO_UF_UNIDADE_SAUDE,267473,1,,267473
CO_UF_RESIDENCIA,267473,1,33,267473
CO_UF_PREST_SERVICO,267473,1,,267473
CO_MUN_UNIDADE_SAUDE,267473,1,,267473
CO_MUN_RESIDENCIA,267473,92,330455,83171
CO_MUN_PREST_SERVICO,267473,1,,267473
CO_UNIDADE_SAUDE,267473,1,,267473
CO_PREST_SERVICO,267473,1,,267473
NU_ANO_COMPETENCIA,267473,1,2025,267473


#### Item 2 - Dicionario operacional da base SISCAN/Mamografia

A base contem exames de mamografia de residentes do Rio de Janeiro em 2025. As colunas estao separadas nos mesmos grupos usados no desenho inicial da Parte 1. Cada tabela informa o tipo lido no Parquet, se a coluna pode ser usada como dado de triagem/inicio do atendimento e, quando a cardinalidade e baixa, os valores observados na base. Para colunas de triagem com baixa cardinalidade, tambem e informado o significado operacional de cada valor.

**Criterio usado para triagem:** campos de triagem sao aqueles que tendem a estar disponiveis na entrada, solicitacao ou avaliacao inicial do atendimento. Campos marcados como `Nao` incluem resultados, BI-RADS, recomendacoes, achados finais do laudo, prazos de resultado ou campos administrativos/vazios. Essa separacao evita vazamento de informacao quando a analise futura quiser simular decisao antes do resultado do exame.

**Fontes para rotulos:** Nota Tecnica 8 do SISCAN/DATASUS para variaveis de mamografia e Manual preliminar do SISCAN/INCA para tipos de mamografia de rastreamento. Quando o rotulo oficial do codigo nao foi identificado no material consultado, isso esta indicado explicitamente.

##### Identificacao tecnica

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `CO_SEQ_SISCAN_MAMOGRAFIA_RESID` | Identificador sequencial do registro. | `str` | Sim | 267473 | Alta cardinalidade (267473 valores) |

##### Residencia e competencia

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `CO_UF_RESIDENCIA` | Codigo da UF de residencia da pessoa atendida. | `str` | Sim | 1 | `33` = Rio de Janeiro |
| `SG_UF_RESIDENCIA` | Sigla da UF de residencia. | `str` | Sim | 1 | `RJ` = Rio de Janeiro |
| `CO_MUN_RESIDENCIA` | Codigo IBGE do municipio de residencia. | `str` | Sim | 92 | Alta cardinalidade (92 valores) |
| `NU_ANO_COMPETENCIA` | Ano de competencia do registro. | `str` | Sim | 1 | `2025` = ano de competencia 2025 |
| `NU_ANO_MES_COMPETENCIA` | Ano e mes de competencia do registro. | `str` | Sim | 12 | `202501` = 01/2025; `202502` = 02/2025; `202503` = 03/2025; `202504` = 04/2025; `202505` = 05/2025; `202506` = 06/2025; `202507` = 07/2025; `202508` = 08/2025; `202509` = 09/2025; `202510` = 10/2025; `202511` = 11/2025; `202512` = 12/2025 |
| `CO_ANO_RESULTADO` | Ano do resultado/laudo. | `str` | Nao | 6 | `2014`, `2016`, `2022`, `2023`, `2024`, `2025` |
| `CO_ANO_MES_RESULTADO` | Ano e mes do resultado/laudo. | `str` | Nao | 30 | Alta cardinalidade (30 valores) |

##### Perfil da pessoa atendida

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `CO_RACA_COR` | Codigo de raca/cor. | `str` | Sim | 6 | `01` = Branca; `02` = Preta; `03` = Parda; `04` = Amarela; `05` = Indigena; `99` = Sem informacao/ignorado |
| `CO_ETNIA` | Codigo de etnia, quando informado. | `str` | Sim | 7 | `(vazio)` = sem preenchimento; `00` = codigo de etnia observado; rotulo nao identificado no material consultado; `01` = codigo de etnia observado; rotulo nao identificado no material consultado; `02` = codigo de etnia observado; rotulo nao identificado no material consultado; `X2` = codigo de etnia observado; rotulo nao identificado no material consultado; `X3` = codigo de etnia observado; rotulo nao identificado no material consultado; `X4` = codigo de etnia observado; rotulo nao identificado no material consultado |
| `CO_IDADE_PACIENTE` | Idade da pessoa atendida. | `str` | Sim | 97 | Alta cardinalidade (97 valores) |
| `CO_ESCOLARIDADE` | Codigo de escolaridade, quando informado. | `str` | Sim | 1 | `(vazio)` = sem preenchimento |
| `SG_SEXO` | Sexo da pessoa atendida. | `str` | Sim | 3 | `F` = Feminino; `I` = Ignorado/indeterminado; `M` = Masculino |

##### Risco, historico e indicacao

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `TP_RESP_APRES_RISC_ELEV_CANCER` | Resposta sobre risco elevado para cancer de mama. | `str` | Sim | 3 | `01` = Sim; `02` = Nao; `03` = Nao sabe |
| `TP_RESP_ANT_MAMA_EXA_PROF_SAUD` | Resposta sobre exame profissional anterior das mamas. | `str` | Sim | 3 | `01` = Sim; `02` = Nao; `03` = Nao sabe |
| `TP_RESP_FEZ_MAMOGRA_ALGUMA_VEZ` | Resposta sobre mamografia previa. | `str` | Sim | 3 | `01` = Sim; `02` = Nao; `03` = Nao sabe |
| `CO_IND_CLINICA` | Indicacao clinica informada na solicitacao/atendimento. | `str` | Sim | 2 | `01` = Mamografia diagnostica; `02` = Mamografia de rastreamento |
| `TP_MAMOGRAFIA_RASTREAMENT` | Classificacao da mamografia como rastreamento ou outra situacao. | `str` | Sim | 4 | `(vazio)` = nao se aplica/sem preenchimento; geralmente exame diagnostico; `01` = Populacao-alvo; `02` = Populacao de risco elevado por historia familiar; `03` = Paciente ja tratado de cancer de mama |

##### Exame clinico/imagem e lateralidade

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `TP_MAMA_PELE_ESQ` | Informacao sobre pele da mama esquerda. | `str` | Sim | 3 | `(vazio)` = nao se aplica/sem preenchimento; `01` = Normal; `02` = Alterada |
| `TP_MAMA_PELE_DIR` | Informacao sobre pele da mama direita. | `str` | Sim | 3 | `(vazio)` = nao se aplica/sem preenchimento; `01` = Normal; `02` = Alterada |
| `TP_MAMA_ESQ` | Caracteristica da mama esquerda apresentada na mamografia. | `str` | Sim | 7 | `(vazio)` = nao se aplica/sem preenchimento; `01` = Densa; `02` = Adiposa; `03` = Predominantemente densa; `04` = Predominantemente adiposa; `05` = Parenquima deslocado anteriormente pelo implante; `06` = Mama reconstruida |
| `TP_MAMA_DIR` | Caracteristica da mama direita apresentada na mamografia. | `str` | Sim | 7 | `(vazio)` = nao se aplica/sem preenchimento; `01` = Densa; `02` = Adiposa; `03` = Predominantemente densa; `04` = Predominantemente adiposa; `05` = Parenquima deslocado anteriormente pelo implante; `06` = Mama reconstruida |
| `TP_LINFONODO_ESQ` | Caracteristica dos linfonodos axilares esquerdos conforme mamografia. | `str` | Sim | 4 | `(vazio)` = nao se aplica/sem preenchimento; `01` = Normal; `02` = Alterado; `03` = Nao visualizado |
| `TP_LINFONODO_DIR` | Caracteristica dos linfonodos axilares direitos conforme mamografia. | `str` | Sim | 4 | `(vazio)` = nao se aplica/sem preenchimento; `01` = Normal; `02` = Alterado; `03` = Nao visualizado |

##### Recomendacoes e BI-RADS

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `TP_RECOMENDACAO_ESQUERDA` | Recomendacao final para mama esquerda. | `str` | Nao | 8 | `(vazio)`, `1`, `2`, `3`, `4`, `5`, `6`, `7` |
| `TP_RECOMENDACAO_DIREITA` | Recomendacao final para mama direita. | `str` | Nao | 8 | `(vazio)`, `1`, `2`, `3`, `4`, `5`, `6`, `7` |
| `CO_RECOMENDACAO` | Codigo da recomendacao final do exame. | `str` | Nao | 8 | `0`, `1`, `2`, `3`, `4`, `5`, `6`, `7` |
| `CO_BIRADS` | Categoria BI-RADS do resultado da mamografia. | `str` | Nao | 7 | `1`, `2`, `3`, `4`, `5`, `6`, `7` |

##### Achados e indicadores complementares

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `NU_CONTROLE_CATEGORIA3` | Indicador/quantidade de controle BI-RADS categoria 3. | `str` | Nao | 2 | `(vazio)`, `1` |
| `NU_LES_DIAG_CANCER` | Indicador/quantidade de lesao com diagnostico de cancer. | `str` | Nao | 2 | `(vazio)`, `1` |
| `NU_DIAG_AVAL_QT` | Indicador/quantidade relacionada a avaliacao diagnostica. | `str` | Nao | 2 | `0`, `1` |
| `NU_REV_MAMOGRAFIA` | Indicador/quantidade de revisao de mamografia. | `str` | Nao | 2 | `0`, `1` |
| `NU_LES_BIOSPIA_PAAF` | Indicador/quantidade de lesao encaminhada a biopsia/PAAF. | `str` | Nao | 2 | `(vazio)`, `1` |
| `NU_NAO_FEZ_CIRURGIAS` | Indicador/quantidade de nao realizacao de cirurgias. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_FEZ_CIRURGIA_ESQ` | Indicador/quantidade de cirurgia na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_FEZ_CIRURGIA_DIR` | Indicador/quantidade de cirurgia na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_NODULO_DENS_GORD_ME` | Indicador/quantidade de nodulo de densidade gordurosa na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_NODULO_CALCIFIC_ME` | Indicador/quantidade de nodulo calcificado na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_NODULO_DEN_HETEROG_ME` | Indicador/quantidade de nodulo de densidade heterogenea na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_CALC_VASCULARES_ME` | Indicador/quantidade de calcificacoes vasculares na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_CALC_TIPICA_BENIG_ME` | Indicador/quantidade de calcificacao tipicamente benigna na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_LINFON_INTRAMAM_ME` | Indicador/quantidade de linfonodo intramamario na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_DIS_ARQ_POR_CIR_ME` | Indicador/quantidade de distorcao arquitetural por cirurgia na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_IMPLANT_SEM_SIN_RUPTU_ME` | Indicador/quantidade de implante sem sinal de ruptura na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_IMP_COM_SIN_RUPTU_ME` | Indicador/quantidade de implante com sinal de ruptura na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_CISTO_OLEOSO_ME` | Indicador/quantidade de cisto oleoso na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_GINECOMASTIA_ME` | Indicador/quantidade de ginecomastia na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_ECTASIA_DUCTAL_ME` | Indicador/quantidade de ectasia ductal na mama esquerda. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_NODULO_DENS_GORD_MD` | Indicador/quantidade de nodulo de densidade gordurosa na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_NODULO_CALCIFIC_MD` | Indicador/quantidade de nodulo calcificado na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_NODULO_DEN_HETEROG_MD` | Indicador/quantidade de nodulo de densidade heterogenea na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_CALC_VASCULARES_MD` | Indicador/quantidade de calcificacoes vasculares na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_CALC_TIPICA_BENIG_MD` | Indicador/quantidade de calcificacao tipicamente benigna na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_LINFON_INTRAMAM_MD` | Indicador/quantidade de linfonodo intramamario na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_DIS_ARQ_POR_CIR_MD` | Indicador/quantidade de distorcao arquitetural por cirurgia na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_IMPLANT_SEM_SIN_RUPTU_MD` | Indicador/quantidade de implante sem sinal de ruptura na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_IMP_COM_SIN_RUPTU_MD` | Indicador/quantidade de implante com sinal de ruptura na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_CISTO_OLEOSO_MD` | Indicador/quantidade de cisto oleoso na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_GINECOMASTIA_MD` | Indicador/quantidade de ginecomastia na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `NU_ECTASIA_DUCTAL_MD` | Indicador/quantidade de ectasia ductal na mama direita. | `str` | Nao | 3 | `(vazio)`, `0`, `1` |
| `CO_TAMANHO_NOD_ESQ` | Tamanho do nodulo na mama esquerda registrado no exame/laudo. | `str` | Nao | 5 | `(vazio)`, `1`, `2`, `3`, `4` |
| `CO_TAMANHO_NOD_DIR` | Tamanho do nodulo na mama direita registrado no exame/laudo. | `str` | Nao | 5 | `(vazio)`, `1`, `2`, `3`, `4` |
| `NU_DIAG_ACHADOS` | Indicador/quantidade de achados diagnosticos. | `str` | Nao | 2 | `0`, `1` |
| `CO_TEMPO_MAMO_ANTERIOR` | Tempo desde mamografia anterior. | `str` | Nao | 6 | `(vazio)`, `0`, `1`, `2`, `3`, `4` |
| `NU_TAMANHO_NODULO` | Tamanho de nodulo registrado no exame/laudo. | `str` | Nao | 5 | `0`, `1`, `2`, `3`, `4` |
| `ST_ASSIMETRIA_FOCAL` | Indicador de assimetria focal no resultado. | `str` | Nao | 2 | `N`, `S` |
| `ST_ASSIMETRIA_DIFUSA` | Indicador de assimetria difusa no resultado. | `str` | Nao | 2 | `N`, `S` |
| `ST_DISTORCAO_FOCAL` | Indicador de distorcao focal no resultado. | `str` | Nao | 2 | `N`, `S` |
| `ST_AREA_DENSA` | Indicador de area densa no resultado. | `str` | Nao | 2 | `N`, `S` |
| `ST_ACHADO_BENIGNO` | Indicador de achado benigno no resultado. | `str` | Nao | 2 | `N`, `S` |
| `ST_TEM_NODULO` | Indicador de presenca de nodulo no resultado. | `str` | Nao | 2 | `N`, `S` |
| `TP_NODULO_CAROCO_MAMA` | Tipo/caracteristica de nodulo ou caroco no exame/laudo. | `str` | Nao | 4 | `01`, `02`, `03`, `04` |
| `ST_TEM_MICROCALCIFICACAO` | Indicador de microcalcificacao no resultado. | `str` | Nao | 2 | `N`, `S` |

##### Tempos e intervalos

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `CO_TEMPO_EXAME` | Tempo ate realizacao do exame. | `str` | Nao | 3 | `1`, `2`, `3` |
| `CO_INTERVALO_RESULTADO` | Intervalo ate resultado/laudo. | `str` | Nao | 4 | `1`, `2`, `3`, `4` |
| `CO_INTERVALO_SOLICITACAO` | Intervalo relacionado a solicitacao. | `str` | Nao | 5 | `(vazio)`, `1`, `2`, `3`, `4` |

##### Colunas administrativas ou sem preenchimento nesta base

| Coluna | Descricao | Tipo no Parquet | Triagem? | Cardinalidade | Valores e significados quando baixa cardinalidade |
|---|---|---|---|---:|---|
| `CO_UF_UNIDADE_SAUDE` | Codigo da UF da unidade de saude; vazio nesta base filtrada. | `str` | Nao | 1 | `(vazio)` |
| `CO_UF_PREST_SERVICO` | Codigo da UF do prestador; vazio nesta base filtrada. | `str` | Nao | 1 | `(vazio)` |
| `CO_MUN_UNIDADE_SAUDE` | Codigo do municipio da unidade de saude; vazio nesta base filtrada. | `str` | Nao | 1 | `(vazio)` |
| `CO_MUN_PREST_SERVICO` | Codigo do municipio do prestador; vazio nesta base filtrada. | `str` | Nao | 1 | `(vazio)` |
| `CO_UNIDADE_SAUDE` | Codigo da unidade de saude; vazio nesta base filtrada. | `str` | Nao | 1 | `(vazio)` |
| `CO_PREST_SERVICO` | Codigo do prestador de servico; vazio nesta base filtrada. | `str` | Nao | 1 | `(vazio)` |

Observacao: os valores possiveis listados sao os valores observados no arquivo `SISCAN_MAMOGRAFIA_RJ_2025.parquet`; para validacao formal dos codigos, consulte a documentacao/tabulacoes oficiais do SISCAN.


#### Item 3 - Verificacao de nulos, vazios e consistencia basica

**Neste passo vamos:**

- medir nulos reais e strings vazias;
- identificar colunas sem informacao;
- avaliar duplicidade do identificador principal;
- confirmar o recorte de UF de residencia.


##### 3.1 - Avaliacao de nulos e strings vazias

In [6]:
nulos = dados.isna().sum()
vazios = dados.astype('string').eq('').sum()
qualidade_campos = (
    pd.DataFrame({
        'nulos': nulos,
        'strings_vazias': vazios,
        'total_sem_informacao': nulos + vazios,
        'percentual_sem_informacao': ((nulos + vazios) / len(dados) * 100).round(2),
    })
    .sort_values('total_sem_informacao', ascending=False)
)
qualidade_campos.head(30)

,nulos,strings_vazias,total_sem_informacao,percentual_sem_informacao
CO_UF_UNIDADE_SAUDE,0,267473,267473,100.0
CO_UF_PREST_SERVICO,0,267473,267473,100.0
CO_MUN_UNIDADE_SAUDE,0,267473,267473,100.0
CO_MUN_PREST_SERVICO,0,267473,267473,100.0
CO_UNIDADE_SAUDE,0,267473,267473,100.0
CO_PREST_SERVICO,0,267473,267473,100.0
CO_ESCOLARIDADE,0,267473,267473,100.0
CO_ETNIA,0,267435,267435,99.99
NU_LES_BIOSPIA_PAAF,0,267234,267234,99.91
NU_LES_DIAG_CANCER,0,267124,267124,99.87


##### 3.2 - Colunas totalmente vazias

Colunas 100% vazias tendem a nao contribuir para a analise desta base filtrada e podem ser removidas da versao tratada.

In [7]:
colunas_totalmente_vazias = qualidade_campos.loc[
    qualidade_campos['total_sem_informacao'] == len(dados)
].index.tolist()
colunas_totalmente_vazias

['CO_UF_UNIDADE_SAUDE',
 'CO_UF_PREST_SERVICO',
 'CO_MUN_UNIDADE_SAUDE',
 'CO_MUN_PREST_SERVICO',
 'CO_UNIDADE_SAUDE',
 'CO_PREST_SERVICO',
 'CO_ESCOLARIDADE']

##### 3.3 - Consistencia do identificador e da UF de residencia

In [8]:
resumo_integridade = {
    'linhas': len(dados),
    'colunas': dados.shape[1],
    'ids_unicos': dados['CO_SEQ_SISCAN_MAMOGRAFIA_RESID'].nunique(dropna=False),
    'ids_duplicados': dados['CO_SEQ_SISCAN_MAMOGRAFIA_RESID'].duplicated().sum(),
    'ufs_residencia': sorted(dados['SG_UF_RESIDENCIA'].dropna().unique().tolist()),
    'codigos_uf_residencia': sorted(dados['CO_UF_RESIDENCIA'].dropna().unique().tolist()),
}
resumo_integridade

{'linhas': 267473,
 'colunas': 82,
 'ids_unicos': 267473,
 'ids_duplicados': np.int64(0),
 'ufs_residencia': ['RJ'],
 'codigos_uf_residencia': ['33']}

#### Item 4 - Tratamento e criacao de variaveis auxiliares

Nesta etapa vamos manter os campos originais e acrescentar variaveis derivadas para facilitar as proximas analises.

**Principais decisoes:**

- strings vazias sao convertidas para `NA`;
- colunas 100% vazias sao removidas;
- `CO_TEMPO_MAMO_ANTERIOR` vazio e tratado em bloco proprio: `5` quando `TP_RESP_FEZ_MAMOGRA_ALGUMA_VEZ` indica que a pessoa nunca fez mamografia e `0` quando continuar sem informacao;
- idade, BI-RADS e campos de ano/mes sao convertidos para numerico quando possivel;
- `COMPETENCIA` e `RESULTADO_MES` sao criadas como datas mensais;
- variaveis auxiliares de rastreamento e risco BI-RADS sao criadas sem apagar os codigos originais.


##### 4.1 - Padronizacao de vazios e remocao de colunas sem informacao

In [9]:
dados_tratado = dados.copy()

for coluna in dados_tratado.select_dtypes(include=['object', 'string']).columns:
    dados_tratado[coluna] = dados_tratado[coluna].astype('string').str.strip()
    dados_tratado[coluna] = dados_tratado[coluna].replace('', pd.NA)

colunas_removidas_sem_info = [coluna for coluna in colunas_totalmente_vazias if coluna in dados_tratado.columns]
dados_tratado = dados_tratado.drop(columns=colunas_removidas_sem_info)

print(f'Colunas removidas por ausencia total de informacao: {colunas_removidas_sem_info}')
print(f'Shape original: {dados.shape}')
print(f'Shape apos remocao: {dados_tratado.shape}')

Colunas removidas por ausencia total de informacao: ['CO_UF_UNIDADE_SAUDE', 'CO_UF_PREST_SERVICO', 'CO_MUN_UNIDADE_SAUDE', 'CO_MUN_PREST_SERVICO', 'CO_UNIDADE_SAUDE', 'CO_PREST_SERVICO', 'CO_ESCOLARIDADE']
Shape original: (267473, 82)
Shape apos remocao: (267473, 75)


##### 4.2 - Tratamento especifico de `CO_TEMPO_MAMO_ANTERIOR`

Antes das conversoes numericas, a coluna `CO_TEMPO_MAMO_ANTERIOR` recebe um tratamento isolado porque seus vazios podem ter dois significados diferentes no SISCAN. Para este notebook, os valores possiveis apos o tratamento sao:

| Codigo | Significado adotado |
|---:|---|
| `0` | Nao informado / sem resposta suficiente |
| `1` | Menos de 1 ano desde a mamografia anterior |
| `2` | De 1 a 2 anos desde a mamografia anterior |
| `3` | De 2 a 3 anos desde a mamografia anterior |
| `4` | Mais de 3 anos desde a mamografia anterior |
| `5` | Nunca fez mamografia |

A intencao e transformar vazios em codigos explicitos: `5` quando o cruzamento com `TP_RESP_FEZ_MAMOGRA_ALGUMA_VEZ` indica que a pessoa nunca fez mamografia; `0` quando nao houve resposta suficiente para inferir o tempo anterior.


In [10]:
coluna_tempo_mamo = 'CO_TEMPO_MAMO_ANTERIOR'
coluna_resp_mamo = 'TP_RESP_FEZ_MAMOGRA_ALGUMA_VEZ'

if {coluna_tempo_mamo, coluna_resp_mamo}.issubset(dados_tratado.columns):
    tempo_mamo_vazio = dados_tratado[coluna_tempo_mamo].isna()
    resposta_mamo = dados_tratado[coluna_resp_mamo].astype('string').str.strip().str.casefold()
    nunca_fez_mamografia = resposta_mamo.isin(['02', '2', 'nao', 'não', 'n'])

    preencher_nunca_fez = tempo_mamo_vazio & nunca_fez_mamografia
    preencher_nao_informado = tempo_mamo_vazio & ~nunca_fez_mamografia

    dados_tratado.loc[preencher_nunca_fez, coluna_tempo_mamo] = '5'
    dados_tratado.loc[preencher_nao_informado, coluna_tempo_mamo] = '0'

    resumo_tratamento_tempo_mamo = {
        'vazios_antes': int(tempo_mamo_vazio.sum()),
        'imputados_com_5_nunca_fez': int(preencher_nunca_fez.sum()),
        'imputados_com_0_nao_informado': int(preencher_nao_informado.sum()),
        'vazios_depois': int(dados_tratado[coluna_tempo_mamo].isna().sum()),
    }
else:
    resumo_tratamento_tempo_mamo = 'Colunas necessarias nao encontradas.'

resumo_tratamento_tempo_mamo


{'vazios_antes': 86526,
 'imputados_com_5_nunca_fez': 42298,
 'imputados_com_0_nao_informado': 44228,
 'vazios_depois': 0}

##### 4.3 - Conversoes numericas e temporais

In [11]:
colunas_numericas = [
    'CO_IDADE_PACIENTE',
    'NU_ANO_COMPETENCIA',
    'NU_ANO_MES_COMPETENCIA',
    'CO_ANO_RESULTADO',
    'CO_ANO_MES_RESULTADO',
    'CO_BIRADS',
    'CO_RECOMENDACAO',
    'CO_TEMPO_EXAME',
    'CO_INTERVALO_RESULTADO',
    'CO_INTERVALO_SOLICITACAO',
    'NU_TAMANHO_NODULO',
    'CO_TEMPO_MAMO_ANTERIOR',
]

for coluna in colunas_numericas:
    if coluna in dados_tratado.columns:
        dados_tratado[f'{coluna}_NUM'] = pd.to_numeric(dados_tratado[coluna], errors='coerce')

if 'NU_ANO_MES_COMPETENCIA' in dados_tratado.columns:
    dados_tratado['COMPETENCIA'] = pd.to_datetime(
        dados_tratado['NU_ANO_MES_COMPETENCIA'].astype('string') + '01',
        format='%Y%m%d',
        errors='coerce',
    )

if 'CO_ANO_MES_RESULTADO' in dados_tratado.columns:
    dados_tratado['RESULTADO_MES'] = pd.to_datetime(
        dados_tratado['CO_ANO_MES_RESULTADO'].astype('string') + '01',
        format='%Y%m%d',
        errors='coerce',
    )

dados_tratado[[c for c in ['CO_TEMPO_MAMO_ANTERIOR_NUM','CO_IDADE_PACIENTE_NUM', 'CO_BIRADS_NUM', 'COMPETENCIA', 'RESULTADO_MES'] if c in dados_tratado.columns]].head()

,CO_TEMPO_MAMO_ANTERIOR_NUM,CO_IDADE_PACIENTE_NUM,CO_BIRADS_NUM,COMPETENCIA,RESULTADO_MES
0,5,55,2,2025-10-01,2025-10-01
1,1,54,2,2025-01-01,2025-01-01
2,4,45,2,2025-11-01,2025-11-01
3,0,52,2,2025-10-01,2025-10-01
4,0,40,2,2025-02-01,2025-02-01


##### 4.4 - Variaveis auxiliares para triagem/rastreamento e BI-RADS

A coluna `TP_MAMOGRAFIA_RASTREAMENT` e mantida como codigo original. A variavel `MAMOGRAFIA_RASTREAMENTO` marca operacionalmente os registros com codigo `01`, que e o principal recorte de rastreamento usado nesta base.

A variavel `BIRADS_GRUPO_RISCO` agrupa a categoria BI-RADS em faixas analiticas usuais para exploracao: negativo/benigno, provavelmente benigno, suspeito, malignidade conhecida e outros/sem classificacao.

In [12]:
dados_tratado['SEXO_LABEL'] = dados_tratado['SG_SEXO'].map({
    'F': 'Feminino',
    'M': 'Masculino',
    'I': 'Ignorado/Indefinido',
}).fillna('Nao informado')

dados_tratado['MAMOGRAFIA_RASTREAMENTO'] = dados_tratado['TP_MAMOGRAFIA_RASTREAMENT'].eq('01').fillna(False)
dados_tratado['MAMOGRAFIA_RASTREAMENTO_LABEL'] = np.where(
    dados_tratado['MAMOGRAFIA_RASTREAMENTO'],
    'Rastreamento',
    'Nao classificada como rastreamento',
)

def classificar_birads(valor):
    if pd.isna(valor):
        return 'Sem classificacao'
    valor = int(valor)
    if valor in [1, 2]:
        return 'Negativo/benigno'
    if valor == 3:
        return 'Provavelmente benigno'
    if valor in [4, 5]:
        return 'Suspeito'
    if valor == 6:
        return 'Malignidade conhecida'
    return 'Outros/inconclusivo'

if 'CO_BIRADS_NUM' in dados_tratado.columns:
    dados_tratado['BIRADS_GRUPO_RISCO'] = dados_tratado['CO_BIRADS_NUM'].apply(classificar_birads)
    dados_tratado['BIRADS_SUSPEITO_OU_MALIGNO'] = dados_tratado['CO_BIRADS_NUM'].isin([4, 5, 6])

colunas_alvo = [
    'TP_MAMOGRAFIA_RASTREAMENT',
    'MAMOGRAFIA_RASTREAMENTO',
    'MAMOGRAFIA_RASTREAMENTO_LABEL',
    'CO_BIRADS',
    'CO_BIRADS_NUM',
    'BIRADS_GRUPO_RISCO',
    'BIRADS_SUSPEITO_OU_MALIGNO',
]

dados_tratado[[c for c in colunas_alvo if c in dados_tratado.columns]].head()

,TP_MAMOGRAFIA_RASTREAMENT,MAMOGRAFIA_RASTREAMENTO,MAMOGRAFIA_RASTREAMENTO_LABEL,CO_BIRADS,CO_BIRADS_NUM,BIRADS_GRUPO_RISCO,BIRADS_SUSPEITO_OU_MALIGNO
0,01,True,Rastreamento,2,2,Negativo/benigno,False
1,01,True,Rastreamento,2,2,Negativo/benigno,False
2,01,True,Rastreamento,2,2,Negativo/benigno,False
3,01,True,Rastreamento,2,2,Negativo/benigno,False
4,01,True,Rastreamento,2,2,Negativo/benigno,False


##### 4.5 - Distribuicoes principais apos tratamento

In [13]:
distribuicoes = {
    'sexo': dados_tratado['SEXO_LABEL'].value_counts(dropna=False),
    'rastreamento': dados_tratado['MAMOGRAFIA_RASTREAMENTO_LABEL'].value_counts(dropna=False),
    'birads_grupo': dados_tratado['BIRADS_GRUPO_RISCO'].value_counts(dropna=False),
    'competencia': dados_tratado['COMPETENCIA'].dt.to_period('M').value_counts().sort_index(),
}

distribuicoes

{'sexo': SEXO_LABEL
 Feminino               266775
 Masculino                 697
 Ignorado/Indefinido         1
 Name: count, dtype: int64,
 'rastreamento': MAMOGRAFIA_RASTREAMENTO_LABEL
 Rastreamento                          238917
 Nao classificada como rastreamento     28556
 Name: count, dtype: int64,
 'birads_grupo': BIRADS_GRUPO_RISCO
 Provavelmente benigno    157555
 Negativo/benigno          98648
 Suspeito                  10421
 Malignidade conhecida       616
 Outros/inconclusivo         233
 Name: count, dtype: int64,
 'competencia': COMPETENCIA
 2025-01    24051
 2025-02    22409
 2025-03    24571
 2025-04    24108
 2025-05    22298
 2025-06    24124
 2025-07    24897
 2025-08    22134
 2025-09    24467
 2025-10    28682
 2025-11    25578
 2025-12      154
 Freq: M, Name: count, dtype: int64}

#### Item 5 - Validacao e gravacao da base tratada

**Neste passo vamos:**

- conferir shape final;
- listar colunas criadas;
- gravar o arquivo tratado em Parquet;
- validar a leitura do arquivo gerado.


##### 5.1 - Shape final e colunas criadas

In [14]:
colunas_criadas = [coluna for coluna in dados_tratado.columns if coluna not in dados.columns]

print(f'Shape original: {dados.shape}')
print(f'Shape tratado: {dados_tratado.shape}')
print('Colunas criadas:')
for coluna in colunas_criadas:
    print(f'- {coluna}')

Shape original: (267473, 82)
Shape tratado: (267473, 94)
Colunas criadas:
- CO_IDADE_PACIENTE_NUM
- NU_ANO_COMPETENCIA_NUM
- NU_ANO_MES_COMPETENCIA_NUM
- CO_ANO_RESULTADO_NUM
- CO_ANO_MES_RESULTADO_NUM
- CO_BIRADS_NUM
- CO_RECOMENDACAO_NUM
- CO_TEMPO_EXAME_NUM
- CO_INTERVALO_RESULTADO_NUM
- CO_INTERVALO_SOLICITACAO_NUM
- NU_TAMANHO_NODULO_NUM
- CO_TEMPO_MAMO_ANTERIOR_NUM
- COMPETENCIA
- RESULTADO_MES
- SEXO_LABEL
- MAMOGRAFIA_RASTREAMENTO
- MAMOGRAFIA_RASTREAMENTO_LABEL
- BIRADS_GRUPO_RISCO
- BIRADS_SUSPEITO_OU_MALIGNO


##### 5.2 - Validacoes finais

In [15]:
validacao_final = {
    'linhas_preservadas': len(dados_tratado) == len(dados),
    'ids_duplicados': dados_tratado['CO_SEQ_SISCAN_MAMOGRAFIA_RESID'].duplicated().sum(),
    'ufs_residencia': sorted(dados_tratado['SG_UF_RESIDENCIA'].dropna().unique().tolist()),
    'colunas_removidas_sem_info': colunas_removidas_sem_info,
    'qtd_colunas_criadas': len(colunas_criadas),
}
validacao_final

{'linhas_preservadas': True,
 'ids_duplicados': np.int64(0),
 'ufs_residencia': ['RJ'],
 'colunas_removidas_sem_info': ['CO_UF_UNIDADE_SAUDE',
  'CO_UF_PREST_SERVICO',
  'CO_MUN_UNIDADE_SAUDE',
  'CO_MUN_PREST_SERVICO',
  'CO_UNIDADE_SAUDE',
  'CO_PREST_SERVICO',
  'CO_ESCOLARIDADE'],
 'qtd_colunas_criadas': 19}

##### 5.3 - Gravacao do Parquet tratado

In [16]:
arquivo_saida = Path('SISCAN_MAMOGRAFIA_RJ_2025_tratado.parquet')
dados_tratado.to_parquet(arquivo_saida, index=False)

validacao_leitura = pd.read_parquet(arquivo_saida)
print(f'Arquivo gerado: {arquivo_saida}')
print(f'Shape gravado: {validacao_leitura.shape}')
print(f'Linhas preservadas no arquivo gravado: {len(validacao_leitura) == len(dados_tratado)}')

Arquivo gerado: SISCAN_MAMOGRAFIA_RJ_2025_tratado.parquet
Shape gravado: (267473, 94)
Linhas preservadas no arquivo gravado: True


---

## Resultado da Parte 1

Ao executar este notebook, a base SISCAN/Mamografia original e tratada e gravada em:

```text
SISCAN_MAMOGRAFIA_RJ_2025_tratado.parquet
```

Resultado validado:

- Base original: 267473 linhas e 82 colunas.
- Base tratada: 267473 linhas e 93 colunas.
- Linhas preservadas apos o tratamento.
- Variaveis auxiliares criadas para apoiar as etapas de analise, selecao de features e modelagem.

Esse arquivo deve ser usado nas proximas etapas de analise exploratoria e preparacao para algoritmos classificatorios.
